# MOSAIC · Yale HER2 treatment response

Shared WSI embeddings → L2 normalization → training-fold standardization → five-fold logistic regression. The 85-patient cohort and seed-42 split are checked. No survival risk score or clinical response field is used as a model input.

Run cells in order. Edit the configuration cell before inference. Install `requirements-notebooks.txt` in this notebook kernel environment first.

## Download the example data

Get [`example_yale-her2_feats/`](https://drive.google.com/drive/folders/1LdLp6TbvtldLgoyjyM9oaKNPjm-1csNF) and `yale_her2_dataset.json` from the [MOSAIC Google Drive folder](https://drive.google.com/drive/folders/1IUbXoDB1KSfdNoSZMUHderIqItNFMSA1?usp=drive_link). Download and extract any ZIP archives, then place the feature folder and JSON directly under the repository's `data/` directory:

```text
data/
├── yale_her2_dataset.json
└── example_yale-her2_feats/
    └── *.h5
```

Keep `.h5` filenames unchanged and avoid an extra nested feature folder. You can use another location by editing `FEATURES` and `METADATA` below. Run the validation cell before inference to check the cohort.


In [ ]:
import copy
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Jupyter can start in the repository root or in notebooks/.
ROOT = Path.cwd()
if not (ROOT / "examples").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "examples" / "reproduce_yale_her2.py").is_file():
    raise FileNotFoundError("Start Jupyter from the MOSAIC repository or notebooks/ directory.")
sys.path.insert(0, str(ROOT))
from examples import reproduce_yale_her2 as workflow


## 1 · Configure paths

Use `Astaxanthin/MOSAIC` or a downloaded MOSAIC directory. KEEP is the patch encoder, not the model to load here. CPU is a portable default; choose CUDA if your PyTorch installation supports it.

In [ ]:
MODEL = "Astaxanthin/MOSAIC"  # Or a local model folder
FEATURES = ROOT / "data/example_yale-her2_feats"
METADATA = ROOT / "data/yale_her2_dataset.json"
OUTPUT = ROOT / "outputs/yale_her2"
DEVICE = "cpu"
REVISION = "main"  # Prefer a commit SHA for a published reproduction
REFERENCE = None  # Optional original all-dataset predictions JSON


## 2 · Validate the cohort

This step checks metadata and feature availability without loading the model or running inference.

In [ ]:
cli = ["--no-interactive", "--model", MODEL, "--revision", REVISION,
       "--features", str(FEATURES), "--metadata", str(METADATA),
       "--output", str(OUTPUT), "--device", DEVICE,
       "--feature-method", "treatment", "--view-batch-size", "8"]
if REFERENCE:
    cli.extend(["--reference", str(REFERENCE)])

args = workflow.parse_args(cli)
check = copy.copy(args)
check.validate_only = True
display(workflow.run(check))


## 3 · Run the reproduction

The first run extracts WSI features and may take several minutes on CPU. Progress is printed. Completed slides are cached; rerunning with the same configuration resumes them. Select a new output directory if you change the model or evaluation settings.

In [ ]:
result = workflow.run(args)

## 4 · Inspect results

In [ ]:
folds = pd.read_csv(Path(args.output) / "fold_metrics.csv")
predictions = pd.read_csv(Path(args.output) / "oof_predictions.csv")
display(folds)
display(predictions.head(10))
print(f"Mean fold AUROC: {result['auc_mean']:.6f}")
print(f"Pooled OOF AUROC: {result['oof_auc']:.6f}")
if "reference_oof_auc" in result:
    print(f"Reference OOF AUROC: {result['reference_oof_auc']:.6f}")
    print(f"Changed predicted classes: {result['changed_predictions']}")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(folds["fold"], folds["auc"], color="#28658f", width=0.6)
ax.axhline(result["auc_mean"], color="#d57827", ls="--", label="Mean fold AUROC")
ax.set(xlabel="Test fold", ylabel="AUROC", ylim=(0, 1), xticks=range(1, 6), title="Yale HER2 · treatment response")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()
